In [ ]:
# CORSIKA showers: sanity checks of the shower tables (make tables)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ev = {"main set": pd.read_feather("tables/finecore.feather"),
      "lighter thinning": pd.read_feather("tables/thinh.feather")}
for d in ev.values():
    d["E_eV"] = d.E_GeV * 1e9

In [ ]:
# Summary
for name, d in ev.items():
    print(f"{name}:  {len(d)} showers at {d.E_eV.nunique()} energies, {d.E_eV.min():.2g}-{d.E_eV.max():.2g} eV;  "
          f"primary {sorted(d.primary_id.unique().tolist())}, zenith {sorted(d.zenith_deg.unique().tolist())} deg;  "
          f"stored muons per fine cell (median) {d.fine_nstored_median.median():g}")

In [ ]:
# Showers per energy
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, d in ev.items():
    n = d.groupby("E_eV").size()
    ax.step(n.index, n.values, where="mid", label=name)
ax.set_xscale("log"); ax.set_ylim(0, None)
ax.set_xlabel("E (eV)"); ax.set_ylabel("showers")
ax.set_title("Showers per energy")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Muons per shower: median and 16-84% band
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, d in ev.items():
    g = d.groupby("E_eV").n_muons
    p = ax.plot(g.median().index, g.median().values, label=name)
    ax.fill_between(g.median().index, g.quantile(0.16).values, g.quantile(0.84).values,
                    color=p[0].get_color(), alpha=0.3, linewidth=0)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("E (eV)"); ax.set_ylabel("muons per shower")
ax.set_title("Muon number")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Thinning: stored muons per fine-core cell (median over the shower's cells, then over showers;
# zero below about 1e16 eV, where most cells are empty)
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, d in ev.items():
    m = d.groupby("E_eV").fine_nstored_median.median()
    ax.plot(m.index, m.values, label=name)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("E (eV)"); ax.set_ylabel("stored muons per fine cell")
ax.set_title("Thinning in the core")
ax.legend(); plt.tight_layout(); plt.show()